# Spark 4.3 features, live data

Same sources as notebook 01. The SCD2 dimension is maintained by the declarative pipeline in `pipelines/meter_cdc` (run every minute by the `batch` service).

In [1]:
import time
from pyspark.sql import functions as F
from lab import *

def timed(df):
    df.limit(1000).collect()
    t0 = time.time(); df.collect(); return round(time.time() - t0, 2)

spark = spark_session("nb-spark-4-3", cores="8", spark__sql__shuffle__partitions="2")
spark.version

'4.3.0'

## 1. SCD Type 2 with Auto CDC

`pipelines/meter_cdc/transformations/dim_meter_scd2.sql` - the whole dimension load is this statement:

In [2]:
print(open("/work/pipelines/meter_cdc/transformations/dim_meter_scd2.sql").read())

CREATE STREAMING TABLE lake.energy.dim_meter_history;

CREATE FLOW dim_meter_history_cdc AS AUTO CDC INTO lake.energy.dim_meter_history
FROM STREAM(meter_changes)
KEYS (meter_id)
APPLY AS DELETE WHEN op = 'DELETE'
SEQUENCE BY seq
COLUMNS * EXCEPT (op, changed_at)
STORED AS SCD TYPE 2
TRACK HISTORY ON (tariff, status, customer, lat, lon);



Target catalog: Auto CDC needs a DSv2 table that supports row-level operations (MERGE). The pipeline writes to **Apache Iceberg**
(`lake` catalog, Hadoop file catalog under `/data/lakehouse`). Iceberg has no official Spark 4.3 build yet, so the image uses the Apache nightly
`iceberg-spark-runtime-4.2` snapshot, which works on 4.3 in testing. Runs are **incremental**: the Kafka checkpoint plus the persistent Iceberg tables
mean each run only applies new change events. Every run is an Iceberg snapshot:

In [3]:
spark.sql("SELECT committed_at, snapshot_id, operation, summary['added-records'] AS added, summary['deleted-records'] AS deleted "
          "FROM lake.energy.dim_meter_history.snapshots ORDER BY committed_at DESC LIMIT 8").show(truncate=False)

+-----------------------+-------------------+---------+-----+-------+
|committed_at           |snapshot_id        |operation|added|deleted|
+-----------------------+-------------------+---------+-----+-------+
|2026-11-06 09:34:38.639|3773825154192714773|overwrite|54   |48     |
|2026-11-06 09:31:54.012|1116638783373558627|overwrite|48   |41     |
|2026-11-06 09:29:11.353|6178091105243292395|append   |41   |NULL   |
+-----------------------+-------------------+---------+-----+-------+



In [4]:
hist = jdbc_read(spark, "dim_meter_history")
versions = hist.groupBy("meter_id").agg(F.count("*").alias("versions"), F.max(F.col("is_current").cast("int")).alias("live"))
versions.orderBy(F.desc("versions")).show(5)
top = versions.orderBy(F.desc("versions")).first().meter_id
hist.where(F.col("meter_id") == top).orderBy("start_seq") \
    .select("tariff", "status", "customer", "valid_from", "valid_to", "is_current").show(truncate=False)

+--------+--------+----+
|meter_id|versions|live|
+--------+--------+----+
|MTR-0023|       2|   1|
|MTR-0011|       2|   1|
|MTR-0024|       2|   1|
|MTR-0002|       2|   1|
|MTR-0003|       2|   1|
+--------+--------+----+
only showing top 5 rows


+------+------+---------+--------------------------+--------------------------+----------+
|tariff|status|customer |valid_from                |valid_to                  |is_current|
+------+------+---------+--------------------------+--------------------------+----------+
|flat  |active|Sam Singh|2026-01-01 00:00:00       |2026-11-06 09:32:32.137824|false     |
|demand|active|Sam Singh|2026-11-06 09:32:32.137824|NULL                      |true      |
+------+------+---------+--------------------------+--------------------------+----------+



In [5]:
late = kafka_batch(spark, "meter_changes", CHANGE_SCHEMA).where("tariff = 'legacy'")
print("late out-of-order events in Kafka:", late.count())
print("rows where a late 'legacy' event became current:", hist.where("tariff = 'legacy' AND is_current").count())
hist.where("tariff = 'legacy'").select("meter_id", "valid_from", "valid_to", "is_current").show(5, truncate=False)

late out-of-order events in Kafka: 2


rows where a late 'legacy' event became current: 2


+--------+--------------------------+--------+----------+
|meter_id|valid_from                |valid_to|is_current|
+--------+--------------------------+--------+----------+
|MTR-0002|2026-11-06 08:31:52.138058|NULL    |true      |
|MTR-0023|2026-11-06 08:28:47.917898|NULL    |true      |
+--------+--------------------------+--------+----------+



## 2. ASOF JOIN

Price every meter read with the latest AEMO dispatch price at or before the read. Compared with the classic range-join + `ROW_NUMBER()` rewrite.

In [6]:
reads = jdbc_read(spark, "SELECT meter_id, region_id, read_ts, kwh FROM fact_meter_read")
prices = jdbc_read(spark, "fact_price_5min")
reads = (reads.crossJoin(spark.range(60).withColumnRenamed("id", "k"))
         .withColumn("read_ts", F.expr("read_ts - make_dt_interval(0, 0, k * 2)")).drop("k")
         .repartition(8).cache())
prices.cache(); print("reads:", reads.count(), "prices:", prices.count())
reads.createOrReplaceTempView("reads"); prices.createOrReplaceTempView("prices")

old_sql = '''
SELECT meter_id, read_ts, rrp FROM (
  SELECT r.meter_id, r.read_ts, p.rrp,
         ROW_NUMBER() OVER (PARTITION BY r.meter_id, r.read_ts ORDER BY p.interval_start DESC) AS rn
  FROM reads r JOIN prices p ON p.region_id = r.region_id AND p.interval_start <= r.read_ts
) WHERE rn = 1'''
asof_sql = '''
SELECT r.meter_id, r.read_ts, p.rrp
FROM reads r ASOF JOIN prices p
  MATCH_CONDITION (r.read_ts >= p.interval_start)
  ON r.region_id = p.region_id'''

for name, sql in [("range join + window", old_sql), ("ASOF JOIN", asof_sql)]:
    df = spark.sql(sql)
    print(f"{name:20s} rows={df.count():,}  {timed(df.groupBy().count()):5.2f}s")
print("identical results:", spark.sql(old_sql).exceptAll(spark.sql(asof_sql)).count() == 0)

reads: 507720 prices: 125


range join + window  rows=507,720   2.24s


ASOF JOIN            rows=507,720   2.00s


identical results: True


In [7]:
spark.sql(asof_sql).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [meter_id#203, read_ts#214, rrp#210]
   +- SortMergeAsOfJoin [region_id#204], [region_id#207], [read_ts#214], [interval_start#208], (read_ts#214 >= interval_start#208), (read_ts#214 - interval_start#208), Inner
      :- Sort [region_id#204 ASC NULLS FIRST, read_ts#214 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(region_id#204, 2), ENSURE_REQUIREMENTS, [plan_id=3267]
      :     +- InMemoryTableScan [meter_id#203, region_id#204, read_ts#214]
      :           +- InMemoryRelation [meter_id#203, region_id#204, read_ts#214, kwh#206], StorageLevel(disk, memory, deserialized, 1 replicas)
      :                 +- AdaptiveSparkPlan isFinalPlan=true
                           +- == Final Plan ==
                              ResultQueryStage 2
                              +- ShuffleQueryStage 1
                                 +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=305]
       

In [8]:
spark.sql('''
SELECT r.region_id, r.read_ts, p.interval_start AS price_from, p.rrp, p.source
FROM reads r LEFT ASOF JOIN prices p
  MATCH_CONDITION (r.read_ts >= p.interval_start)
  ON r.region_id = p.region_id
ORDER BY r.read_ts DESC LIMIT 8''').show(truncate=False)

+---------+--------------------------+-------------------+-----+------+
|region_id|read_ts                   |price_from         |rrp  |source|
+---------+--------------------------+-------------------+-----+------+
|NSW1     |2026-11-06 09:35:48.208647|2026-11-06 09:30:00|97.09|AEMO  |
|NSW1     |2026-11-06 09:35:48.208647|2026-11-06 09:30:00|97.09|AEMO  |
|NSW1     |2026-11-06 09:35:48.208647|2026-11-06 09:30:00|97.09|AEMO  |
|NSW1     |2026-11-06 09:35:48.208647|2026-11-06 09:30:00|97.09|AEMO  |
|NSW1     |2026-11-06 09:35:48.208647|2026-11-06 09:30:00|97.09|AEMO  |
|NSW1     |2026-11-06 09:35:48.208647|2026-11-06 09:30:00|97.09|AEMO  |
|NSW1     |2026-11-06 09:35:48.208647|2026-11-06 09:30:00|97.09|AEMO  |
|NSW1     |2026-11-06 09:35:48.208647|2026-11-06 09:30:00|97.09|AEMO  |
+---------+--------------------------+-------------------+-----+------+



## 3. Python UDFs transpiled to SQL (experimental)

Requires `spark.sql.experimental.optimizer.transpilePyUDFs=true`, ANSI mode, and a **non-Arrow** batched UDF.
Because 4.2 made Arrow the default, you must pass `useArrow=False` for transpilation to kick in.

In [9]:
spark.conf.set("spark.sql.experimental.optimizer.transpilePyUDFs", "true")
print("ansi:", spark.conf.get("spark.sql.ansi.enabled"))

def meter_code_len(s: str) -> int:
    return len(s)

def wh_to_kwh(wh: float) -> float:
    return wh / 1000.0

candidates = {
    "len() default/Arrow": F.udf(meter_code_len, "int"),
    "len() useArrow=False": F.udf(meter_code_len, "int", useArrow=False),
    "wh/1000 useArrow=False": F.udf(wh_to_kwh, "double", useArrow=False),
}
src = spark.range(3_000_000).selectExpr("concat('MTR-', id) AS s", "rand(7) * 5000 AS wh").cache(); src.count()
for name, u in candidates.items():
    col = "s" if "len" in name else "wh"
    q = src.select(F.sum(u(col)))
    plan = q._jdf.queryExecution().executedPlan().toString()
    dt = timed(q)
    print(f"{name:24s} python in plan: {'EvalPython' in plan!s:5s}  {dt:5.2f}s")

ansi: true


len() default/Arrow      python in plan: True    1.26s


len() useArrow=False     python in plan: False   0.50s


wh/1000 useArrow=False   python in plan: True    1.39s


## 4. JSON_TABLE and in-place VARIANT edits

Device diagnostics arrive as nested JSON. `variant_delete` drops the debug block, `variant_set` stamps lineage, `JSON_TABLE` flattens `readings[*]`.

In [10]:
raw = (spark.read.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("subscribe", "device_events")
       .option("startingOffsets", "earliest").load().selectExpr("parse_json(cast(value AS string)) AS v"))
raw.createOrReplaceTempView("raw_events")
spark.sql('''
SELECT to_json(variant_set(variant_delete(v, '$.debug'), '$.pipeline', 'notebook')) AS cleaned
FROM raw_events LIMIT 1''').show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|cleaned                                                                                                                                                                                                                                                   |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"device":{"firmware":"4.0.0-rc1","id":"DEV-0028","meter_id":"MTR-0028"},"pipeline":"notebook","readings":[{"component":"meter","metrics":{"temp_c":26.4}},{"component":"comms","fault":false,"metrics":{"rssi":-63}}],"ts":"2026-11-06T09:28:07

In [11]:
spark.sql('''
SELECT variant_get(e.v, '$.device.meter_id', 'string') AS meter_id, t.*
FROM raw_events e,
LATERAL JSON_TABLE(to_json(e.v), '$.readings[*]' COLUMNS (
    n          FOR ORDINALITY,
    component  STRING  PATH '$.component',
    temp_c     DOUBLE  PATH '$.metrics.temp_c',
    rssi       INT     PATH '$.metrics.rssi',
    efficiency DOUBLE  PATH '$.metrics.efficiency',
    fault         BOOLEAN PATH '$.fault',
    reports_fault BOOLEAN EXISTS PATH '$.fault'
)) AS t
LIMIT 10''').show()

+--------+---+---------+------+----+----------+-----+-------------+
|meter_id|  n|component|temp_c|rssi|efficiency|fault|reports_fault|
+--------+---+---------+------+----+----------+-----+-------------+
|MTR-0028|  1|    meter|  26.4|NULL|      NULL| NULL|        false|
|MTR-0028|  2|    comms|  NULL| -63|      NULL|false|         true|
|MTR-0007|  1|    meter|  44.2|NULL|      NULL| NULL|        false|
|MTR-0007|  2|    comms|  NULL| -91|      NULL|false|         true|
|MTR-0021|  1|    meter|  43.7|NULL|      NULL| NULL|        false|
|MTR-0021|  2|    comms|  NULL| -60|      NULL|false|         true|
|MTR-0021|  3| inverter|  59.9|NULL|     0.937|false|         true|
|MTR-0008|  1|    meter|  29.8|NULL|      NULL| NULL|        false|
|MTR-0008|  2|    comms|  NULL| -75|      NULL|false|         true|
|MTR-0023|  1|    meter|  40.5|NULL|      NULL| NULL|        false|
+--------+---+---------+------+----+----------+-----+-------------+



In [12]:
spark.sql('''
SELECT t.component, count(*) AS readings, round(avg(t.temp_c), 1) AS avg_temp, sum(cast(t.fault AS int)) AS faults
FROM raw_events e,
LATERAL JSON_TABLE(to_json(e.v), '$.readings[*]' COLUMNS (component STRING, temp_c DOUBLE PATH '$.metrics.temp_c', fault BOOLEAN)) AS t
GROUP BY t.component''').show()

+---------+--------+--------+------+
|component|readings|avg_temp|faults|
+---------+--------+--------+------+
|    meter|     129|    35.0|  NULL|
|    comms|     129|    NULL|    10|
| inverter|       8|    59.8|     0|
|  charger|      13|    45.2|     0|
+---------+--------+--------+------+



## 5. Stateful Real-Time Mode

`dropDuplicates` + windowed aggregation now run in Real-Time Mode on pipelined shuffles. Each updated window is emitted as soon as a record changes it.
Needs enough task slots for every pipelined stage at once (`CONCURRENT_SCHEDULER_INSUFFICIENT_SLOT` otherwise).

In [13]:
q = (kafka_stream(spark, "meter_reads", READ_SCHEMA, starting="latest")
     .withWatermark("read_ts", "10 seconds")
     .dropDuplicates(["meter_id", "read_ts"])
     .groupBy("region_id", F.window("read_ts", "10 seconds").alias("w"))
     .agg(F.count("*").alias("reads"), F.round(F.sum("kwh"), 5).alias("kwh"))
     .selectExpr("region_id AS key", "to_json(named_struct('region_id', region_id, 'window_start', w.start, 'reads', reads, 'kwh', kwh)) AS value")
     .writeStream.format("kafka")
     .option("kafka.bootstrap.servers", KAFKA)
     .option("topic", "nb_rtm_windows")
     .option("checkpointLocation", f"/data/notebooks/ckpt/rtm_state_{int(time.time())}")
     .outputMode("update")
     .trigger(realTime="30 seconds")
     .start())
time.sleep(25)
q.stop()
kafka_batch(spark, "nb_rtm_windows", RTM_WINDOW_SCHEMA).orderBy(F.desc("kafka_ts")).show(10, truncate=False)

+---------+-------------------+-----+-------+-----------------------+
|region_id|window_start       |reads|kwh    |kafka_ts               |
+---------+-------------------+-----+-------+-----------------------+
|TAS1     |2026-11-06 09:37:00|52   |0.07508|2026-11-06 09:37:06.35 |
|QLD1     |2026-11-06 09:37:00|12   |0.00931|2026-11-06 09:37:06.341|
|NSW1     |2026-11-06 09:37:00|36   |0.06484|2026-11-06 09:37:06.341|
|VIC1     |2026-11-06 09:37:00|20   |0.02861|2026-11-06 09:37:06.308|
|TAS1     |2026-11-06 09:37:00|51   |0.07444|2026-11-06 09:37:06.308|
|VIC1     |2026-11-06 09:37:00|19   |0.02695|2026-11-06 09:37:06.307|
|NSW1     |2026-11-06 09:37:00|34   |0.06279|2026-11-06 09:37:06.307|
|TAS1     |2026-11-06 09:37:00|49   |0.07242|2026-11-06 09:37:06.307|
|TAS1     |2026-11-06 09:37:00|50   |0.07309|2026-11-06 09:37:06.307|
|NSW1     |2026-11-06 09:37:00|35   |0.06411|2026-11-06 09:37:06.307|
+---------+-------------------+-----+-------+-----------------------+
only showing top 10 

## 6. Nanosecond timestamps (preview)

In [14]:
spark.conf.set("spark.sql.timestampNanosTypes.enabled", "true")
ns = spark.sql('''
SELECT timestamp_nanos(1762414000123456789) AS ts_ltz9,
       CAST('2026-11-06 10:00:00.123456789' AS TIMESTAMP_NTZ(9)) AS ts_ntz9,
       unix_nanos(CAST('2026-11-06 10:00:00.123456789' AS TIMESTAMP_NTZ(9))) AS nanos''')
ns.printSchema(); ns.show(truncate=False)
ns.write.mode("overwrite").parquet("/data/notebooks/nanos")
spark.read.parquet("/data/notebooks/nanos").show(truncate=False)

root
 |-- ts_ltz9: timestamp_ltz(9) (nullable = false)
 |-- ts_ntz9: timestamp_ntz(9) (nullable = true)
 |-- nanos: decimal(21,0) (nullable = true)



+-----------------------------+-----------------------------+-------------------+
|ts_ltz9                      |ts_ntz9                      |nanos              |
+-----------------------------+-----------------------------+-------------------+
|2025-11-06 07:26:40.123456789|2026-11-06 10:00:00.123456789|1793959200123456789|
+-----------------------------+-----------------------------+-------------------+



+-----------------------------+-----------------------------+-------------------+
|ts_ltz9                      |ts_ntz9                      |nanos              |
+-----------------------------+-----------------------------+-------------------+
|2025-11-06 07:26:40.123456789|2026-11-06 10:00:00.123456789|1793959200123456789|
+-----------------------------+-----------------------------+-------------------+

